<a href="https://colab.research.google.com/github/emyemy89/scalable-ml-id2223/blob/lab2-task12/ID2223_Lab_Session_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Session 3 - Retrieval-Augmented Generation for (Multi-hop) Question Answering

**Student notebook.** Work through the sections in order. The lab instructions doc describes the tasks and the report requirements in full.

* Cells marked **`TODO (Task X)`** are yours to implement. Most of them raise `NotImplementedError` until you do.
* **[REPORT n]** cells are questions you must answer, either in this notebook or in your report.
* The harness code (data loading, metrics, LLM wrapper, result tables) is provided. You may change it, but say so in your report if you do.

**Runtime:** Google Colab, **T4 GPU** (*Runtime → Change runtime type → T4 GPU*). The default sizes are chosen so that the whole notebook runs in one free Colab session.

## 0. Setup

In [1]:
# Installing vLLM takes ~3-5 minutes and replaces Colab's PyTorch build and some other libraries.
# pip will print "ERROR: pip's dependency resolver ... incompatible" for Colab-preinstalled packages
# (google-colab, bigquery, gradio, cudf, ...). This is expected and harmless: the lab doesn't use them.

# TODO AFTER THIS CELL: Runtime -> Restart session, then continue from the next cell (don't re-run this one).
!pip install -q vllm chromadb sentence-transformers rank_bm25 datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompati

In [1]:
import os, re, gc, json, time, random, string, collections
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_colwidth", 120)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

# ---------------- Experiment size ----------------
# Small enough for a free T4 session. Use larger sizes for your final numbers if you have the budget.
N_SQUAD_QUESTIONS = 300    # SQuAD questions evaluated (corpus = all SQuAD dev articles)
N_HOTPOT_POOL = 500        # HotpotQA questions whose 10 paragraphs form the HotpotQA corpus (~5k paragraphs)
N_HOTPOT_QUESTIONS = 200   # HotpotQA questions evaluated (a subset of the pool)

# ---------------- Models ----------------
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"   # faster alternative: "Qwen/Qwen2.5-1.5B-Instruct"
LLM_BACKEND = "vllm"                     # "vllm" (fast, batched) or "hf" (plain transformers fallback, slow)
LLM_MAX_MODEL_LEN = 8192
LLM_GPU_MEMORY_UTILIZATION = 0.6         # leave room on the T4 for the embedding / re-ranking models
MAX_CONTEXT_CHARS = 16000                # retrieved context is truncated to this many characters (~4k tokens)

EMBEDDING_MODELS = {
    "minilm":    "sentence-transformers/all-MiniLM-L6-v2",
    "bge-small": "BAAI/bge-small-en-v1.5",
    "e5-small":  "intfloat/e5-small-v2",
}
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # stronger/slower: "BAAI/bge-reranker-base"

CHROMA_DIR = "/content/chroma" if os.path.isdir("/content") else "./chroma"

Device: cuda Tesla T4


## 1. Data

We use two QA datasets:

| Dataset | Type | Corpus we index | Gold evidence |
|---|---|---|---|
| **SQuAD v1.1** (dev) | single-hop, extractive | all 48 dev Wikipedia articles (paragraphs joined) | the character span of the answer |
| **HotpotQA** (distractor, dev) | multi-hop (bridge & comparison) | the 10 paragraphs (2 gold + 8 distractors) of `N_HOTPOT_POOL` questions | the 2 supporting Wikipedia paragraphs |

**Single-hop vs multi-hop:**

* **Single-hop:** *When was Isaac Newton born?* One document (Newton's page) contains the answer.
* **Multi-hop, bridge:** *In what year was the university where Sergei Aleksandrovich Tokarev was a professor founded?* First you retrieve Tokarev's page to learn it was Moscow State University. Only then can you retrieve the page that says it was founded in 1755. The second document shares almost no words with the question, so single-shot retrieval tends to miss it.
* **Multi-hop, comparison:** *Between Greyia and Calibanus, which genus contains more species?* Both entities are named in the question, so retrieval is easier, but the model has to combine facts from two documents.

Every question carries its **gold evidence** as character spans in the corpus. That lets us measure retrieval quality for *any* chunking strategy: a retrieved chunk counts as a hit if it overlaps a gold span. `gold` is a list of *evidence groups*. SQuAD has 1 group, with the alternative annotated answer spans inside it. HotpotQA has 2 groups, one per supporting paragraph. Full retrieval success means every group is hit.

*(Optional extension: 2WikiMultiHopQA has the same format as HotpotQA. See <https://github.com/Alab-NII/2wikimultihop>.)*

In [2]:
from datasets import load_dataset

@dataclass
class Doc:
    doc_id: str
    title: str
    text: str

@dataclass
class Question:
    qid: str
    question: str
    answers: list      # acceptable answer strings
    gold: list         # evidence groups: [[(doc_id, start, end), ...alternatives], ...]
    evidence: list     # gold passages (doc_id, start, end), used for the oracle-context ablation
    dataset: str
    qtype: str = ""

WHOLE_DOC = 10**9  # 'end' offset meaning "until the end of the document"


def load_squad(n_questions, seed=SEED):
    ds = load_dataset("rajpurkar/squad", split="validation")
    articles = collections.OrderedDict()
    for title, ctx in zip(ds["title"], ds["context"]):
        paras = articles.setdefault(title, [])
        if not paras or paras[-1] != ctx:
            if ctx not in paras:
                paras.append(ctx)
    docs, para_pos = [], {}
    for i, (title, paras) in enumerate(articles.items()):
        doc_id, text = f"squad-{i}", ""
        for p in paras:
            if text:
                text += "\n\n"
            para_pos[p] = (doc_id, len(text))
            text += p
        docs.append(Doc(doc_id, title.replace("_", " "), text))
    questions = []
    for i in random.Random(seed).sample(range(len(ds)), n_questions):
        ex = ds[i]
        doc_id, off = para_pos[ex["context"]]
        spans = sorted({(doc_id, off + s, off + s + len(a))
                        for s, a in zip(ex["answers"]["answer_start"], ex["answers"]["text"])})
        questions.append(Question(ex["id"], ex["question"], list(dict.fromkeys(ex["answers"]["text"])),
                                  gold=[spans], evidence=[(doc_id, off, off + len(ex["context"]))],
                                  dataset="squad"))
    return docs, questions


def load_hotpot(n_pool, n_questions, seed=SEED):
    ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
    docs, title2id, questions = [], {}, []
    for j, i in enumerate(random.Random(seed).sample(range(len(ds)), n_pool)):
        ex = ds[i]
        for title, sents in zip(ex["context"]["title"], ex["context"]["sentences"]):
            if title not in title2id:
                title2id[title] = f"hotpot-{len(docs)}"
                docs.append(Doc(title2id[title], title, "".join(sents).strip()))
        if j < n_questions:
            gold_titles = list(dict.fromkeys(ex["supporting_facts"]["title"]))
            spans = [(title2id[t], 0, WHOLE_DOC) for t in gold_titles]
            questions.append(Question(ex["id"], ex["question"], [ex["answer"]], gold=[[s] for s in spans],
                                      evidence=spans, dataset="hotpot", qtype=ex["type"]))
    return docs, questions


squad_docs, squad_qs = load_squad(N_SQUAD_QUESTIONS)
hotpot_docs, hotpot_qs = load_hotpot(N_HOTPOT_POOL, N_HOTPOT_QUESTIONS)
CORPORA = {"squad": squad_docs, "hotpot": hotpot_docs}
EVAL_SETS = {"squad": squad_qs, "hotpot": hotpot_qs}
DOCS = {d.doc_id: d for docs in CORPORA.values() for d in docs}

for name, docs in CORPORA.items():
    lens = [len(d.text) for d in docs]
    print(f"{name:7s} corpus: {len(docs):5d} docs, {sum(lens)/1e6:.2f}M chars, "
          f"mean {np.mean(lens):.0f} chars/doc | {len(EVAL_SETS[name])} eval questions")
print("HotpotQA question types:", collections.Counter(q.qtype for q in hotpot_qs))

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

... (7 outputs left out)

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

squad   corpus:    48 docs, 1.60M chars, mean 33429 chars/doc | 300 eval questions
hotpot  corpus:  4955 docs, 2.70M chars, mean 546 chars/doc | 200 eval questions
HotpotQA question types: Counter({'bridge': 170, 'comparison': 30})


In [3]:
def gold_text(q, max_chars=300):
    out = []
    for doc_id, s, e in q.evidence:
        d = DOCS[doc_id]
        out.append(f"  [{d.title}] {d.text[s:min(e, len(d.text))][:max_chars]}...")
    return "\n".join(out)

for q in (squad_qs[0], hotpot_qs[0], hotpot_qs[1]):
    print(f"[{q.dataset}/{q.qtype or 'single-hop'}] Q: {q.question}\n  A: {q.answers}\n  Evidence:\n{gold_text(q)}\n")

[squad/single-hop] Q: How might gravity effects be observed differently according to Newton?
  A: ['at larger distances.', 'at larger distances']
  Evidence:
  [Force] Newton came to realize that the effects of gravity might be observed in different ways at larger distances. In particular, Newton determined that the acceleration of the Moon around the Earth could be ascribed to the same force of gravity if the acceleration due to gravity decreased as an inverse sq...

[hotpot/bridge] Q: In what year was the university where Sergei Aleksandrovich Tokarev was a professor founded?
  A: ['1755']
  Evidence:
  [Sergei Aleksandrovich Tokarev] Sergei Aleksandrovich Tokarev (Russian: Серге́й Алекса́ндрович То́карев , 29 December 1899 – 19 April 1985) was a Russian scholar, ethnographer, historian, researcher of religious beliefs, doctor of historical sciences, and professor at Moscow State University....
  [Moscow State University] Lomonosov Moscow State University (MSU; Russian: Московский го

## 2. Chunking (Task 1)

A chunker maps a document's text to a list of **`(start, end)` character spans**. The chunk text is `text[start:end]`. Keeping offsets (rather than only strings) lets us check gold-evidence overlap and fetch neighbouring chunks later.

`chunk_paragraphs` is given. Implement:
* **`chunk_sentences(text, n_sentences, overlap)`**: groups of `n_sentences` consecutive sentences. Consecutive chunks share `overlap` sentences.
* **`chunk_fixed(text, size, overlap)`**: windows of about `size` characters, with `overlap` characters shared between consecutive chunks. Avoid cutting words in half.

The test cell below checks that your spans are valid and cover all of the text.

In [4]:
def chunk_paragraphs(text):
    """One chunk per paragraph (paragraphs are separated by blank lines)."""
    spans, start = [], 0
    for m in re.finditer(r"\n\s*\n", text):
        if text[start:m.start()].strip():
            spans.append((start, m.start()))
        start = m.end()
    if text[start:].strip():
        spans.append((start, len(text)))
    return spans


_SENT_BOUNDARY = re.compile(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])")

def sentence_spans(text):
    """Rough rule-based sentence splitter returning (start, end) spans."""
    spans, start = [], 0
    for m in _SENT_BOUNDARY.finditer(text):
        spans.append((start, m.start()))
        start = m.end()
    spans.append((start, len(text)))
    return [(s, e) for s, e in spans if text[s:e].strip()]

In [9]:
def chunk_sentences(text, n_sentences=3, overlap=1):
    """TODO (Task 1): groups of `n_sentences` sentences; consecutive chunks share `overlap` sentences.
    Hint: use `sentence_spans(text)` and merge spans of consecutive sentences. Don't emit a trailing chunk
    that is fully contained in the previous one."""
    spans = sentence_spans(text)
    if not spans:
        return []
    assert n_sentences > 0 and 0 <= overlap < n_sentences
    result = []
    step = n_sentences - overlap
    for i in range(0, len(spans), step):
        group = spans[i:i + n_sentences]
        if not group:
            break
        start = group[0][0]
        end = group[-1][1]
        if result and end <= result[-1][1]:
            break
        result.append((start, end))
    return result


def chunk_fixed(text, size=1000, overlap=200):
    """TODO (Task 1): windows of ~`size` characters where consecutive windows share ~`overlap` characters.
    Avoid cutting words in half (e.g. end the window at the last whitespace) and make sure the loop always
    makes progress."""
    if not text.strip():
        return []
    assert size > 0 and 0 <= overlap < size
    result = []
    start = 0
    step = size - overlap
    while start < len(text):
        end = min(start + size, len(text))
        # Move the end to a word boundary
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start:
                end = boundary
        result.append((start, end))
        if end == len(text):
            break
        # Start the next chunk with the requested overlap
        next_start = max(start + 1, end - overlap)
        # Move backwards to avoid starting in the middle of a word
        while (next_start > start and
               not text[next_start].isspace() and
               not text[next_start - 1].isspace()):
            next_start -= 1

        start = next_start
    return result


In [10]:
CHUNKERS = {
    "paragraph":      chunk_paragraphs,
    "sent1":          lambda t: chunk_sentences(t, 1, 0),
    "sent3_ov1":      lambda t: chunk_sentences(t, 3, 1),
    "char300_ov50":   lambda t: chunk_fixed(t, 300, 50),
    "char1000_ov0":   lambda t: chunk_fixed(t, 1000, 0),
    "char1000_ov200": lambda t: chunk_fixed(t, 1000, 200),
}

def check_chunker(fn, text):
    spans = fn(text)
    assert spans and all(0 <= s < e <= len(text) for s, e in spans), "invalid spans"
    covered = np.zeros(len(text), bool)
    for s, e in spans:
        covered[s:e] = True
    missing = [i for i, c in enumerate(text) if not c.isspace() and not covered[i]]
    assert not missing, f"{len(missing)} non-space chars not covered, e.g. at {missing[:5]}"

for name, fn in CHUNKERS.items():
    for d in squad_docs[:5] + hotpot_docs[:50]:
        check_chunker(fn, d.text)
print("All chunkers OK")

rows = []
for ds_name, docs in CORPORA.items():
    for name, fn in CHUNKERS.items():
        lens = [e - s for d in docs for s, e in fn(d.text)]
        rows.append(dict(dataset=ds_name, chunker=name, n_chunks=len(lens), mean_chars=np.mean(lens),
                         p95_chars=np.percentile(lens, 95)))
pd.DataFrame(rows).round(0)

All chunkers OK


,dataset,chunker,n_chunks,mean_chars,p95_chars
0,squad,paragraph,2067,774.0,1338.0
1,squad,sent1,10222,156.0,316.0
2,squad,sent3_ov1,5098,468.0,783.0
3,squad,char300_ov50,6615,295.0,299.0
4,squad,char1000_ov0,1633,983.0,999.0
5,squad,char1000_ov200,2037,985.0,999.0
6,hotpot,paragraph,4955,546.0,1119.0
7,hotpot,sent1,21508,125.0,253.0
8,hotpot,sent3_ov1,9879,333.0,575.0
9,hotpot,char300_ov50,12425,249.0,299.0


## 3. Embeddings and the Chroma vector database (Task 2)

We embed chunks with small `sentence-transformers` models and store them in **Chroma**. We compute the embeddings ourselves (rather than letting Chroma do it) so that we control the model, the batching and the query/passage prefixes that some models expect.

Implement:
* **`build_index`**: chunk the corpus, embed the chunks, and store them in a Chroma collection with cosine distance.
* **`dense_retrieve`**: embed a batch of queries and return the top-k chunks for each.

In [11]:
from sentence_transformers import SentenceTransformer
import chromadb

class Embedder:
    # Some models were trained with instruction prefixes; using them matters for retrieval quality.
    QUERY_PREFIX = {"intfloat/e5-small-v2": "query: ",
                    "BAAI/bge-small-en-v1.5": "Represent this sentence for searching relevant passages: "}
    DOC_PREFIX = {"intfloat/e5-small-v2": "passage: "}

    def __init__(self, name):
        self.name = name
        self.model = SentenceTransformer(name, device=DEVICE)

    def _encode(self, texts, prefix, batch_size=128):
        return self.model.encode([prefix + t for t in texts], batch_size=batch_size, normalize_embeddings=True,
                                 convert_to_numpy=True, show_progress_bar=len(texts) > 5000)

    def embed_docs(self, texts):
        return self._encode(texts, self.DOC_PREFIX.get(self.name, ""))

    def embed_queries(self, texts):
        return self._encode(texts, self.QUERY_PREFIX.get(self.name, ""))

_EMBEDDERS = {}
def get_embedder(key):
    if key not in _EMBEDDERS:
        _EMBEDDERS[key] = Embedder(EMBEDDING_MODELS[key])
    return _EMBEDDERS[key]

chroma = chromadb.PersistentClient(path=CHROMA_DIR)

@dataclass
class Index:
    name: str
    collection: object
    chunks: list          # chunk dicts: id, doc_id, title, chunk_idx, start, end, text
    by_id: dict           # chunk id -> chunk dict
    embedder: Embedder
    docs: dict            # doc_id -> Doc
    build_seconds: float = 0.0

In [14]:
def build_index(name, docs, chunker, embedder, add_title=True):
    """TODO (Task 2): chunk `docs`, embed the chunks and store them in a (fresh) Chroma collection `name`.

    1. For every doc, call `chunker(doc.text)` and create one chunk dict per span with keys
       id=f"{doc_id}::{chunk_idx}", doc_id, title, chunk_idx, start, end, text.
    2. Embed the chunk texts with `embedder.embed_docs(...)`. If `add_title`, prepend the doc title
       to the text. (Why might that help? Think of chunks that start with "He ..." or "The band ...".)
    3. Delete any existing collection called `name`, then create a new one that uses cosine distance
       (`metadata={"hnsw:space": "cosine"}`, `embedding_function=None`), and add ids, embeddings,
       documents and metadatas. Chroma limits the batch size, so add a few thousand chunks at a time.
    4. Return Index(name, collection, chunks, by_id, embedder, {doc_id: doc}, build_seconds).
    """
    start_time = time.perf_counter()
    chunks = []
    by_id = {}
    # 1. Split documents into chunks
    for doc in docs:
        spans = chunker(doc.text)
        for chunk_idx, (start, end) in enumerate(spans):
            chunk = {
                "id": f"{doc.doc_id}::{chunk_idx}",
                "doc_id": doc.doc_id,
                "title": doc.title,
                "chunk_idx": chunk_idx,
                "start": start,
                "end": end,
                "text": doc.text[start:end]
            }
            chunks.append(chunk)
    # Create a lookup dictionary
    by_id = {chunk["id"]: chunk for chunk in chunks}
    # Delete an existing collection with the same name
    existing = chroma.list_collections()
    existing_names = [
        c.name if hasattr(c, "name") else c
        for c in existing
    ]
    if name in existing_names:
        chroma.delete_collection(name)
    # 4. Create a fresh collection using cosine distance
    collection = chroma.create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"},
        embedding_function=None
    )
    # 5. Embed and store chunks in batches
    batch_size = 2000
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]

        texts = [
            f"{chunk['title']}\n{chunk['text']}"
            if add_title else chunk["text"]
            for chunk in batch
        ]
        embeddings = embedder.embed_docs(texts)
        collection.add(
            ids=[chunk["id"] for chunk in batch],
            embeddings=embeddings.tolist(),
            documents=[chunk["text"] for chunk in batch],
            metadatas=[
                {
                    "doc_id": chunk["doc_id"],
                    "title": chunk["title"],
                    "chunk_idx": chunk["chunk_idx"],
                    "start": chunk["start"],
                    "end": chunk["end"]
                }
                for chunk in batch
            ]
        )
    build_seconds = time.perf_counter() - start_time
    # 6. Return the complete index
    return Index(
        name,
        collection,
        chunks,
        by_id,
        embedder,
        {doc.doc_id: doc for doc in docs},
        build_seconds
    )


def dense_retrieve(index, queries, k):
    """TODO (Task 2): for each query string return the top-k chunks, best first.
    Each result should be a copy of the chunk dict (from `index.by_id`) with an added 'score' field
    (cosine similarity = 1 - cosine distance). Embed all queries in one batch and send them to
    `collection.query` in one call."""
    if not queries:
        return []
    if not index.chunks:
        return [[] for _ in queries]
    # 1. Embed all queries together
    query_embeddings = index.embedder.embed_queries(queries)
    # 2. Search Chroma in one call
    results = index.collection.query(
        query_embeddings=query_embeddings.tolist(),
        n_results=k,
        include=["distances"]
    )
    # 3. Convert results into lists of chunk dictionaries
    retrieved = []
    for ids, distances in zip(results["ids"], results["distances"]):
        query_results = []
        for chunk_id, distance in zip(ids, distances):
            chunk = index.by_id[chunk_id].copy()
            # Convert cosine distance into cosine similarity
            chunk["score"] = 1 - distance
            query_results.append(chunk)
        retrieved.append(query_results)
    return retrieved

In [15]:
_idx = build_index("smoke-test", squad_docs[:3], chunk_paragraphs, get_embedder("minilm"))
for c in dense_retrieve(_idx, ["When was the Super Bowl played?"], k=3)[0]:
    print(f"{c['score']:.3f}  {c['id']:12s} {c['text'][:100]}")
chroma.delete_collection("smoke-test")

0.690  squad-0::7   On May 21, 2013, NFL owners at their spring meetings in Boston voted and awarded the game to Levi's 
0.680  squad-0::0   Super Bowl 50 was an American football game to determine the champion of the National Football Leagu
0.651  squad-0::28  The game's media day, which was typically held on the Tuesday afternoon prior to the game, was moved


## 4. Retrieval: experiments and evaluation (Task 3)

**Metrics** (computed against the gold evidence spans):
* `recall@k`: fraction of evidence groups hit by at least one of the top-k chunks.
* `all@k`: fraction of questions where *all* groups are hit. For HotpotQA this means both supporting paragraphs were retrieved.
* `mrr`: reciprocal rank of the first relevant chunk.
* `ctx_chars@5`: how much text the top-5 chunks contain. **Larger chunks trivially get higher recall at a fixed k**, so compare configurations at a similar context budget as well as at a similar k.

In [ ]:
KS = (1, 3, 5, 10, 20)

def span_hit(chunk, span):
    doc_id, s, e = span
    return chunk["doc_id"] == doc_id and chunk["start"] < e and chunk["end"] > s

def groups_hit(q, chunks):
    return [any(span_hit(c, sp) for c in chunks for sp in group) for group in q.gold]

def retrieval_metrics(questions, retrieved, ks=KS):
    rows = []
    for q, hits in zip(questions, retrieved):
        row = {}
        for k in ks:
            g = groups_hit(q, hits[:k])
            row[f"recall@{k}"], row[f"all@{k}"] = np.mean(g), float(all(g))
        row["mrr"] = next((1 / r for r, c in enumerate(hits, 1)
                           if any(span_hit(c, sp) for grp in q.gold for sp in grp)), 0.0)
        row["ctx_chars@5"] = sum(len(c["text"]) for c in hits[:5])
        rows.append(row)
    return pd.DataFrame(rows).mean().to_dict()

### 4.1 Chunking × embedding model grid

The cell below builds one index per (dataset, chunker, embedder) and evaluates dense retrieval. Edit the lists to choose what you compare. You must compare **at least 3 chunking strategies** (including overlap vs no overlap) and **at least 2 embedding models**.

In [ ]:
GRID_CHUNKERS = ["paragraph", "sent1", "sent3_ov1", "char300_ov50", "char1000_ov0", "char1000_ov200"]
GRID_EMBEDDERS = ["minilm", "bge-small", "e5-small"]

grid = []
for ds_name, docs in CORPORA.items():
    qs = EVAL_SETS[ds_name]
    for emb_key in GRID_EMBEDDERS:
        for ch_key in GRID_CHUNKERS:
            idx = build_index(f"grid-{ds_name}-{ch_key}-{emb_key}", docs, CHUNKERS[ch_key], get_embedder(emb_key))
            t0 = time.perf_counter()
            hits = dense_retrieve(idx, [q.question for q in qs], k=max(KS))
            ms_per_q = 1000 * (time.perf_counter() - t0) / len(qs)
            grid.append(dict(dataset=ds_name, embedder=emb_key, chunker=ch_key, n_chunks=len(idx.chunks),
                             build_s=idx.build_seconds, query_ms=ms_per_q, **retrieval_metrics(qs, hits)))
            chroma.delete_collection(idx.name)   # free memory; we rebuild the chosen config below

grid_df = pd.DataFrame(grid)
grid_df.round(3).sort_values(["dataset", "recall@5"], ascending=[True, False])

> **[REPORT 1]** Which chunking strategy and embedding model work best for each dataset, and why? Discuss (a) chunk size versus recall at a fixed k, and versus recall at a fixed **context budget**; (b) whether overlap helps; (c) differences between embedding models, including cost (`build_s`) and the role of query/passage prefixes; (d) why `all@k` is much lower than `recall@k` on HotpotQA.

*Your answer:*



In [ ]:
# TODO (Task 3): choose the configuration you will use for the rest of the lab, one per dataset,
# and justify the choice in the report.
CHOSEN = {
    "squad":  dict(chunker="...", embedder="..."),
    "hotpot": dict(chunker="...", embedder="..."),
}

In [ ]:
INDEX = {ds: build_index(f"main-{ds}", CORPORA[ds], CHUNKERS[cfg["chunker"]], get_embedder(cfg["embedder"]))
         for ds, cfg in CHOSEN.items()}
for ds, idx in INDEX.items():
    print(f"{ds}: {len(idx.chunks)} chunks, built in {idx.build_seconds:.1f}s")

### 4.2 Choosing k, and retrieving surrounding chunks

Small chunks are precise but can lose context. One fix is to retrieve with small chunks and then **expand each hit with its neighbouring chunks** (the `window` chunks before and after it in the same document), merging overlapping passages.

In [ ]:
def expand_with_neighbors(index, hits, window=1):
    """TODO (Task 3): replace each hit by the passage covering chunks [chunk_idx - window, chunk_idx + window]
    of the same document (clip at the document boundaries). Merge passages from the same document that
    overlap, and keep the rank order of the hits. Each returned passage is a chunk-like dict with
    doc_id, title, start, end, text (text = doc.text[start:end])."""
    raise NotImplementedError

In [ ]:
rows = []
for ds, idx in INDEX.items():
    qs = EVAL_SETS[ds]
    hits = dense_retrieve(idx, [q.question for q in qs], k=20)
    for k in (1, 2, 3, 5, 10, 20):
        top = [h[:k] for h in hits]
        for window in (0, 1, 2):
            ctx = top if window == 0 else [expand_with_neighbors(idx, h, window) for h in top]
            m = retrieval_metrics(qs, ctx, ks=(k,))
            rows.append(dict(dataset=ds, k=k, window=window, recall=m[f"recall@{k}"], all_gold=m[f"all@{k}"],
                             ctx_chars=np.mean([sum(len(c["text"]) for c in x) for x in ctx])))
k_df = pd.DataFrame(rows)
k_df.pivot_table(index=["dataset", "k"], columns="window", values=["all_gold", "ctx_chars"]).round(3)

> **[REPORT 2]** Which k would you use, with and without neighbour expansion, and why? Base the justification on the recall versus context-length trade-off above. (You will revisit this choice in Task 4, using answer quality.)

*Your answer:*



## 5. The reader LLM and the evaluation harness (given)

We serve the LLM with **vLLM**, which batches all questions into one call and makes evaluating hundreds of questions fast. `chat()` is the only function that talks to the LLM. It records the tokens used and the time spent generating.

> If vLLM does not start on your runtime, set `LLM_BACKEND = "hf"` in the config cell. That fallback uses plain `transformers` and works everywhere, but it is much slower.

In [ ]:
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

if LLM_BACKEND == "vllm":
    os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")   # run the engine in-process (notebook friendly)
    from vllm import LLM, SamplingParams
    llm = LLM(model=LLM_NAME, dtype="half",   # T4 has no bfloat16 support
              max_model_len=LLM_MAX_MODEL_LEN, gpu_memory_utilization=LLM_GPU_MEMORY_UTILIZATION, seed=SEED)
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    hf_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
    hf_model = AutoModelForCausalLM.from_pretrained(
        LLM_NAME, dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()

TOKENS = collections.Counter()   # prompt / completion tokens used so far
PHASE = collections.Counter()    # seconds spent in "generation"


def _hf_chat(convs, max_tokens, temperature, batch_size=8):
    outs = []
    for i in range(0, len(convs), batch_size):
        texts = [hf_tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True)
                 for c in convs[i:i + batch_size]]
        enc = hf_tok(texts, return_tensors="pt", padding=True).to(hf_model.device)
        with torch.no_grad():
            gen = hf_model.generate(**enc, max_new_tokens=max_tokens, do_sample=temperature > 0,
                                    temperature=temperature if temperature > 0 else None, top_p=None, top_k=None,
                                    pad_token_id=hf_tok.pad_token_id or hf_tok.eos_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        TOKENS["prompt"] += int(enc["attention_mask"].sum())
        TOKENS["completion"] += int((new != (hf_tok.pad_token_id or hf_tok.eos_token_id)).sum())
        outs += hf_tok.batch_decode(new, skip_special_tokens=True)
    return outs


def chat(prompts, system=None, max_tokens=64, temperature=0.0):
    """Batched chat completion: one user message per prompt -> list of response strings."""
    if not prompts:
        return []
    convs = [([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": p}]
             for p in prompts]
    t0 = time.perf_counter()
    if LLM_BACKEND == "vllm":
        outs = llm.chat(convs, SamplingParams(max_tokens=max_tokens, temperature=temperature, seed=SEED),
                        use_tqdm=False)
        TOKENS["prompt"] += sum(len(o.prompt_token_ids) for o in outs)
        TOKENS["completion"] += sum(len(o.outputs[0].token_ids) for o in outs)
        texts = [o.outputs[0].text for o in outs]
    else:
        texts = _hf_chat(convs, max_tokens, temperature)
    PHASE["generation"] += time.perf_counter() - t0
    return [t.strip() for t in texts]

print(chat(["What is the capital of Sweden? Answer in one word."]))

### Prompts, answer metrics and the experiment runner

* **Answer metrics:** SQuAD-style **EM** and token **F1** after normalisation, plus a lenient **Acc**: the normalised gold answer appears in the normalised prediction. Small instruction-tuned models are often correct but verbose, which EM punishes.
* A **system** is a function `answer_fn(questions) -> (predictions, contexts[, traces])`. `contexts[i]` is the list of chunks that were shown to the LLM for question `i`, and is used to score retrieval.
* **`run_system(name, questions, answer_fn)`** runs a system on a batch, scores it, and stores it in `RESULTS`.
* **`summarize()`** builds the comparison table, including *end-to-end* numbers: accuracy conditioned on whether retrieval found all the evidence, and the fraction of questions where the pipeline was right *and* the evidence was retrieved.
* **`measure_latency()`** answers questions **one at a time**, to measure what a single user would experience.
* **GPU-hours** = wall-clock time of the batched run × 1 GPU. We also extrapolate to the full dev set.

In [ ]:
SYSTEM_QA = ("You are a question answering system. Reply with the answer only: a short phrase (usually 1-5 words), "
             "no full sentence and no explanation.")
TRUNCATIONS = collections.Counter()

def format_context(chunks, max_chars=MAX_CONTEXT_CHARS):
    parts, total = [], 0
    for i, c in enumerate(chunks, 1):
        block = f"[{i}] {c['title']}\n{c['text']}"
        if total + len(block) > max_chars:
            TRUNCATIONS["contexts"] += 1
            break
        parts.append(block)
        total += len(block)
    return "\n\n".join(parts) if parts else "(no passages)"

def prompt_closed_book(question):
    return f"Question: {question}\nAnswer:"

def prompt_rag(question, chunks):
    return (f"Answer the question using the context passages below. If they are insufficient, give your best guess."
            f"\n\nContext:\n{format_context(chunks)}\n\nQuestion: {question}\nAnswer:")

def clean_answer(text):
    lines = [l for l in text.strip().split("\n") if l.strip()]
    text = lines[0] if lines else ""
    text = re.sub(r"^\s*(final answer|answer)\s*:\s*", "", text, flags=re.I)
    return text.strip().strip("*").strip().rstrip(".").strip()

# --- SQuAD-style answer metrics ---
def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, golds):
    return float(any(normalize_answer(pred) == normalize_answer(g) for g in golds))

def f1_score(pred, golds):
    def f1(p, g):
        p, g = normalize_answer(p).split(), normalize_answer(g).split()
        common = sum((collections.Counter(p) & collections.Counter(g)).values())
        if common == 0:
            return 0.0
        prec, rec = common / len(p), common / len(g)
        return 2 * prec * rec / (prec + rec)
    return max(f1(pred, g) for g in golds)

def lenient_match(pred, golds):
    p = f" {normalize_answer(pred)} "
    return float(any(normalize_answer(g) and f" {normalize_answer(g)} " in p for g in golds))

In [ ]:
RESULTS = {}
FULL_TEST_SIZE = {"squad": 10570, "hotpot": 7405}   # dev-set sizes, for GPU-hour extrapolation

def run_system(name, questions, answer_fn):
    gen0, tok0, trunc0 = PHASE["generation"], TOKENS.copy(), TRUNCATIONS["contexts"]
    t0 = time.perf_counter()
    out = answer_fn(questions)
    seconds = time.perf_counter() - t0
    preds, contexts = out[0], out[1]
    traces = out[2] if len(out) > 2 else [None] * len(questions)
    rows = []
    for q, p, ctx, tr in zip(questions, preds, contexts, traces):
        g = groups_hit(q, ctx)
        rows.append(dict(qid=q.qid, dataset=q.dataset, qtype=q.qtype, question=q.question, gold=q.answers, pred=p,
                         em=exact_match(p, q.answers), f1=f1_score(p, q.answers), acc=lenient_match(p, q.answers),
                         ret_recall=np.mean(g) if ctx else np.nan, ret_all=float(all(g)) if ctx else np.nan,
                         n_ctx=len(ctx), ctx_chars=sum(len(c["text"]) for c in ctx), contexts=ctx, trace=tr))
    df = pd.DataFrame(rows)
    RESULTS[name] = dict(rows=df, seconds=seconds, gen_seconds=PHASE["generation"] - gen0,
                         prompt_tokens=TOKENS["prompt"] - tok0["prompt"],
                         completion_tokens=TOKENS["completion"] - tok0["completion"])
    msg = f"{name:32s} EM={df.em.mean():.3f} F1={df.f1.mean():.3f} Acc={df.acc.mean():.3f}"
    if df.ret_all.notna().any():
        msg += f" ret_all={df.ret_all.mean():.3f}"
    msg += f" | {seconds:.1f}s"
    if TRUNCATIONS["contexts"] > trunc0:
        msg += f" | WARNING: {TRUNCATIONS['contexts'] - trunc0} contexts truncated"
    print(msg)
    return RESULTS[name]


def measure_latency(name, questions, answer_fn, n=20):
    """Answer `n` questions one by one; store p50/p95 latency (ms) for system `name`."""
    tot, gen = [], []
    for q in questions[:n]:
        g0, t0 = PHASE["generation"], time.perf_counter()
        answer_fn([q])
        tot.append(1000 * (time.perf_counter() - t0))
        gen.append(1000 * (PHASE["generation"] - g0))
    RESULTS[name]["latency"] = {"p50 ms": np.median(tot), "p95 ms": np.percentile(tot, 95),
                                "p50 gen ms": np.median(gen)}
    return RESULTS[name]["latency"]


def summarize(prefix=""):
    rows = []
    for name, r in RESULTS.items():
        if not name.startswith(prefix):
            continue
        df, n = r["rows"], len(r["rows"])
        ds = df.dataset.iloc[0]
        has_ret = df.ret_all.notna().any()
        hit, miss = df.ret_all == 1, df.ret_all == 0
        rows.append({
            "system": name, "n": n, "EM": df.em.mean(), "F1": df.f1.mean(), "Acc": df.acc.mean(),
            "ret recall": df.ret_recall.mean(), "ret all-gold": df.ret_all.mean(),
            "Acc | all-gold hit": df.acc[hit].mean() if has_ret else np.nan,
            "Acc | miss": df.acc[miss].mean() if has_ret else np.nan,
            "E2E (hit & correct)": (hit & (df.acc == 1)).mean() if has_ret else np.nan,
            "ctx chars": df.ctx_chars.mean(),
            "tokens/q": (r["prompt_tokens"] + r["completion_tokens"]) / n,
            "batched s/q": r["seconds"] / n,
            "GPU-h (run)": r["seconds"] / 3600,
            "GPU-h (full dev, extrap.)": r["seconds"] / 3600 * FULL_TEST_SIZE[ds] / n,
            **r.get("latency", {}),
        })
    return pd.DataFrame(rows).set_index("system").round(4)

## 6. Question answering systems (Task 4)

### 6.1 LLM-only baseline (given)

In [ ]:
def closed_book_system():
    def answer_fn(qs):
        preds = chat([prompt_closed_book(q.question) for q in qs], system=SYSTEM_QA, max_tokens=32)
        return [clean_answer(p) for p in preds], [[] for _ in qs]
    return answer_fn

def rag_system(retrieve):
    """Single-shot RAG. retrieve(list[Question]) -> list[list[chunk]] (the contexts shown to the LLM)."""
    def answer_fn(qs):
        ctxs = retrieve(qs)
        preds = chat([prompt_rag(q.question, c) for q, c in zip(qs, ctxs)], system=SYSTEM_QA, max_tokens=32)
        return [clean_answer(p) for p in preds], ctxs
    return answer_fn

for ds, qs in EVAL_SETS.items():
    run_system(f"{ds}/closed_book", qs, closed_book_system())

### 6.2 RAG with top-k dense retrieval

Use `rag_system` with a retriever built on your `INDEX[ds]`. Choose k (and optionally a neighbour window) **based on answer quality**: run a few values and justify your choice.

In [ ]:
# TODO (Task 4.1): write `dense_retriever(ds, k, window=0)` that returns a retrieve(qs) function for rag_system,
# then run RAG for several values of k (and optionally a neighbour window) on both datasets, e.g.
#   run_system(f"{ds}/dense_k{k}", qs, rag_system(dense_retriever(ds, k)))
# Finally set K to your chosen value.

K = ...
summarize()

> **[REPORT 3]** Which k did you choose, and why? Discuss the answer-quality vs context-length vs latency trade-off. Does more context ever *hurt*?

*Your answer:*



### 6.3 RAG with re-ranking

Retrieve a larger candidate set (e.g. 20–30 chunks) with dense retrieval, re-score each (query, chunk) pair with a **cross-encoder**, and keep the top `K`.

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder(RERANKER_NAME, device=DEVICE)

def rerank(queries, candidates, k):
    """TODO (Task 4.2): queries: list[str]; candidates: list[list[chunk]].
    Score all (query, title + chunk text) pairs with `reranker.predict(...)` in ONE batched call and return
    the top-k chunks per query (best first)."""
    raise NotImplementedError

# TODO (Task 4.2): build a retriever that takes the top-N dense candidates and re-ranks them down to K,
# then run it on both datasets, e.g. run_system(f"{ds}/rerank_k{K}", qs, rag_system(...)).

### 6.4 RAG with query rewriting

Users' questions are not always good search queries. Ask the LLM to rewrite the question into a search query, then retrieve with it. Consider whether to keep the original question too (e.g. by fusing the two rankings), and whether to generate several queries.

In [ ]:
REWRITE_PROMPT = """Rewrite the question below as a concise search query for a Wikipedia search engine.
Keep all names, dates and key terms; drop filler words. Output only the query.

Question: {question}
Search query:"""

def clean_line(text):
    line = next((l for l in text.strip().split("\n") if l.strip()), "")
    line = re.sub(r"^\s*(search query|query)\s*:\s*", "", line, flags=re.I)
    return line.strip().strip('"').strip()

def rrf(rankings, k, c=60):
    """TODO (Task 4.3/4.4): Reciprocal Rank Fusion for ONE query: score(chunk) = sum over rankings of 1/(c + rank).
    rankings: list of ranked chunk lists. Return the top-k fused chunks (deduplicated by chunk id)."""
    raise NotImplementedError

# TODO (Task 4.3): implement query rewriting with `chat` (batched!) and a retriever that uses the rewritten
# query, with or without the original question. Run it on both datasets and inspect some rewritten queries.

### 6.5 Hybrid retrieval (BM25 + dense)

Dense retrieval is good at paraphrase but weak at rare names and numbers, which lexical BM25 handles well. Build a BM25 index over the **same chunks** and fuse the two rankings (e.g. with Reciprocal Rank Fusion).

In [ ]:
from rank_bm25 import BM25Okapi

def bm25_tokenize(text):
    return re.findall(r"\w+", text.lower())

class BM25Index:
    """TODO (Task 4.4): a BM25 index over the SAME chunks as a dense Index (index.chunks),
    with a batched `retrieve(queries, k)` that returns chunk dicts (with 'score'), best first."""
    def __init__(self, index):
        raise NotImplementedError

    def retrieve(self, queries, k):
        raise NotImplementedError

# TODO (Task 4.4): build BM25 = {ds: BM25Index(INDEX[ds])}, then a hybrid retriever that fuses dense and BM25
# rankings (e.g. with your `rrf`). Run BM25-only and hybrid systems on both datasets.

## 7. Evaluation (Task 5)

### 7.1 Numerical evaluation

Run the latency measurement for every system you report, then build the summary table. Columns:
* **Answer quality**: EM / F1 / Acc.
* **Retrieval quality**: `ret recall`, `ret all-gold` (on the contexts actually shown to the LLM).
* **End-to-end**: `Acc | all-gold hit`, `Acc | miss`, `E2E (hit & correct)`.
* **Latency**: p50/p95 for single questions. **Throughput**: batched s/q.
* **Cost**: tokens per question, GPU-hours for this run and extrapolated to the full dev set.

In [ ]:
# TODO (Task 5.1): call measure_latency(name, qs, answer_fn) for every system you report, e.g.
#   measure_latency(f"{ds}/dense_k{K}", qs, rag_system(dense_retriever(ds, K)))

summary = summarize()
summary

> **[REPORT 4]** Summarise the numerical results. Which retrieval improvement helps most on SQuAD, and which on HotpotQA? Why? What does `Acc | miss` tell you about the reader's parametric knowledge? What do latency and GPU cost add to the picture: is the best system worth its cost?

*Your answer:*



### 7.2 Qualitative error analysis

Pick **at least 3 incorrect answers** from your best RAG system. For each, inspect the retrieved chunks (below) and classify the error as:
1. **Retrieval error**: the evidence was not retrieved.
2. **Reasoning/reading error**: the evidence was retrieved, but the model answered wrongly.
3. **Long-context error**: the evidence was retrieved but buried among many passages. The model gets it right with less context (check this with the oracle ablation in 7.3).
4. **Evaluation artefact**: the answer is actually correct, but phrased differently.

In [ ]:
def show_errors(name, n=3, seed=0, max_chars=250):
    df = RESULTS[name]["rows"]
    wrong = df[df.acc == 0]
    print(f"{name}: {len(wrong)}/{len(df)} wrong (lenient)\n")
    q_by_id = {q.qid: q for qs in EVAL_SETS.values() for q in qs}
    for _, r in wrong.sample(min(n, len(wrong)), random_state=seed).iterrows():
        q = q_by_id[r.qid]
        print("=" * 100)
        print(f"Q: {r.question}\nGOLD: {r.gold}   PRED: {r.pred}   retrieved all gold: {r.ret_all}")
        print(f"Gold evidence:\n{gold_text(q, max_chars)}")
        for i, c in enumerate(r.contexts, 1):
            mark = "GOLD" if any(span_hit(c, sp) for g in q.gold for sp in g) else "    "
            print(f"  [{i}] {mark} {c['title']}: {c['text'][:max_chars]!r}")
        if r.trace:
            print("Trace:")
            for step in r.trace:
                print("   ", step)

# TODO (Task 5.2): inspect errors of your best system(s), e.g. show_errors("hotpot/<your system>", n=5)

> **[REPORT 5]** For each of your (at least) 3 incorrect answers: show the question, gold answer, prediction and the relevant retrieved chunks, and explain *why* the answer was wrong (retrieval vs reasoning vs long context vs evaluation artefact). Then estimate how often each error type occurs, using the conditional accuracies from 7.1.

*Your answer:*



### 7.3 Controlled ablations: retrieval vs answering

Separate the contribution of the retriever from that of the reader:
* **Reader ablation**: keep the reader fixed and vary the context: *none* (closed-book) vs *retrieved* vs *oracle* (the gold evidence passages). The gap between oracle and retrieved is the retrieval loss. The gap between oracle and 100% is the reader's loss.
* **Long-context ablation**: give the reader the gold passage(s) plus *n* distractors (high-scoring but non-gold retrieved chunks). Vary *n* and the **position** of the gold passage (start / middle / end). See "Lost in the Middle" (Liu et al., 2023).

In [ ]:
def oracle_retrieve(qs):
    """The gold evidence passages (SQuAD: the answer paragraph; HotpotQA: the two supporting paragraphs)."""
    out = []
    for q in qs:
        out.append([dict(id=f"oracle::{doc_id}::{s}", doc_id=doc_id, title=DOCS[doc_id].title, chunk_idx=-1,
                         start=s, end=min(e, len(DOCS[doc_id].text)), text=DOCS[doc_id].text[s:e])
                    for doc_id, s, e in q.evidence])
    return out

for ds, qs in EVAL_SETS.items():
    run_system(f"{ds}/oracle", qs, rag_system(oracle_retrieve))

In [ ]:
# TODO (Task 5.3): long-context ablation. Implement a retriever that returns the oracle passage(s) plus
# n_distractors non-gold chunks (e.g. the highest-ranked dense hits that do NOT overlap the gold evidence),
# with the gold passage(s) placed at the start / middle / end. Run it for several n_distractors and positions
# and tabulate the accuracy. Watch for the "contexts truncated" warning: it means the context exceeded
# MAX_CONTEXT_CHARS.

> **[REPORT 6]** What do the two ablations show? Quantify (a) how much accuracy is lost to retrieval vs to the reader, and (b) whether the reader degrades with more context and/or with the position of the evidence. Relate this to your error analysis in 7.2.

*Your answer:*



## 8. Iterative retrieval for multi-hop QA (Task 6)

Single-shot retrieval fails on bridge questions such as *"What government position was held by the woman who portrayed Corliss Archer in Kiss and Tell?"*. The second paragraph (about the actress) can only be found **after** reading the first one.

Implement a system that, for each question:
1. Writes a search query for the first piece of missing information and retrieves.
2. **Synthesises** the retrieved passages into short notes and writes the **next** query.
3. Repeats once more, then answers from the notes and all retrieved passages.

**Batch across questions:** run each hop for all questions in one `chat()` call. Calling the LLM once per question is ~10–50× slower. Return the per-question trace so you can inspect it with `show_errors`.

In [ ]:
FIRST_QUERY_PROMPT = """Question: {question}

Answering this question may require several facts from Wikipedia. Write a search query to find the FIRST fact you need.
Output only the query."""

HOP_PROMPT = """You are answering a multi-hop question step by step with a search engine.

Question: {question}

Notes so far:
{notes}

Newly retrieved passages:
{passages}

1. Update the notes: add the facts from the passages that help answer the question (be brief, keep earlier useful notes).
2. Write the next search query, for the information that is still missing.

Respond exactly in this format:
NOTES: <updated notes>
NEXT QUERY: <search query>"""

FINAL_PROMPT = """Answer the question using the notes and passages.

Question: {question}

Notes:
{notes}

Passages:
{passages}

Answer (short phrase only):"""

def parse_fields(text, fields):
    """Parse 'FIELD: value' sections from an LLM response (case-insensitive; values may span lines)."""
    pattern = r"(?im)^[\s*#]*(" + "|".join(re.escape(f) for f in fields) + r")[\s*]*:\s*"
    parts = re.split(pattern, text)
    return {parts[i].upper(): parts[i + 1].strip().strip("*").strip() for i in range(1, len(parts) - 1, 2)}

def add_unique(ctx, new):
    seen = {(c["doc_id"], c["start"], c["end"]) for c in ctx}
    for c in new:
        if (c["doc_id"], c["start"], c["end"]) not in seen:
            ctx.append(c)
            seen.add((c["doc_id"], c["start"], c["end"]))

In [ ]:
def iterative_rag_system(retrieve_texts, k=3, n_rounds=3):
    """TODO (Task 6): iterative multi-hop RAG.
    retrieve_texts(list[str], k) -> list[list[chunk]] (e.g. dense, hybrid, or hybrid + re-ranking on INDEX["hotpot"]).
    Must return (predictions, contexts, traces), where contexts[i] = all unique chunks retrieved for question i
    and traces[i] = a list of per-round dicts (query, retrieved titles, notes) for inspection.
    Use FIRST_QUERY_PROMPT, HOP_PROMPT, FINAL_PROMPT, parse_fields and add_unique, and batch every LLM call
    across all questions."""
    raise NotImplementedError

# TODO (Task 6): run your system on hotpot_qs (compare with single-shot RAG at a similar number of chunks,
# and vary n_rounds), measure its latency and inspect some traces with show_errors(...).

In [ ]:
summ = summarize("hotpot/")
summ.loc[[i for i in summ.index if not "ablation" in i]]

In [ ]:
def by_qtype(names):
    rows = []
    for n in names:
        df = RESULTS[n]["rows"]
        for t, g in df.groupby("qtype"):
            rows.append(dict(system=n, qtype=t, n=len(g), acc=g.acc.mean(), ret_all=g.ret_all.mean()))
    return pd.DataFrame(rows).pivot_table(index="system", columns="qtype", values=["acc", "ret_all"]).round(3)

# TODO (Task 6): compare your systems per HotpotQA question type, e.g.
# by_qtype([f"hotpot/dense_k{K}", "hotpot/<your iterative system>", "hotpot/oracle"])

> **[REPORT 7]** Does iterative retrieval improve retrieval (all-gold) and answer accuracy over single-shot RAG with a comparable number of retrieved chunks? Break it down by bridge vs comparison questions. What does it cost in latency and tokens? Show one trace where the second query found the bridge paragraph, and one where the loop went wrong.

*Your answer:*



## 10. Final summary

Collect your final comparison table (below) and the plots or tables you reference in your report.

In [ ]:
final = summarize()
final.loc[[i for i in final.index if "ablation" not in i]]